# Fine-tune PaddleOCR's recognizer for OCRead (dot print on caps)

Runtime → Change runtime type → **GPU**. Then run the cells top to bottom.

You need `ocread_rec_dataset.zip`, made on the laptop with `python -m QC_tools.make_rec_dataset`.
At the end you download `ocread_rec_model.zip`; see docs/finetune_ocread.md for using it in the app.

In [ ]:
!nvidia-smi

## 1. Install PaddlePaddle (GPU) and PaddleOCR's training code
If the install fails, get the command for Colab's CUDA version from https://www.paddlepaddle.org.cn/en/install/quick (pip, Linux, GPU) and use that instead.

In [ ]:
!pip install -q paddlepaddle-gpu==3.2.2 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
import paddle; paddle.utils.run_check()

In [ ]:
!git clone -q --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git
%cd PaddleOCR
!pip install -q -r requirements.txt

## 2. Upload the dataset

In [ ]:
from google.colab import files
up = files.upload()                      # choose ocread_rec_dataset.zip
!mkdir -p train_data/ocread && unzip -q -o ocread_rec_dataset.zip -d train_data/ocread
!wc -l train_data/ocread/train.txt train_data/ocread/val.txt && head -3 train_data/ocread/train.txt

## 3. Pretrained PP-OCRv5 mobile recognizer (starting point)

In [ ]:
!mkdir -p pretrain_models && wget -q -nc -P pretrain_models https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv5_mobile_rec_pretrained.pdparams
!ls -la pretrain_models

## 4. Fine-tune
Output alphabet: digits, capital letters and `: / . -` (dict.txt). Letters are needed so embossed "PE/PA" is read as letters (and ignored by the format check) instead of being forced into digits.
Batch size 32 fits a Colab T4 (128 ran out of GPU memory). Training takes a few hours for 20 epochs; you can stop once `acc` stops improving and run the export cell - it exports the best checkpoint so far.
If a setting name is rejected (`KeyError`, e.g. for `sampler`), remove that setting and copy the error to Claude.

In [ ]:
CFG = 'configs/rec/PP-OCRv5/PP-OCRv5_mobile_rec.yml'
COMMON = ('Global.character_dict_path=train_data/ocread/dict.txt Global.use_space_char=True '
          'Global.max_text_length=25')
!python tools/train.py -c {CFG} -o {COMMON} \
    Global.pretrained_model=pretrain_models/PP-OCRv5_mobile_rec_pretrained.pdparams \
    Global.epoch_num=20 Global.save_model_dir=output/ocread_rec Global.eval_batch_step="[0,500]" \
    Optimizer.lr.learning_rate=0.0003 \
    Train.dataset.data_dir=train_data/ocread Train.dataset.label_file_list="['train_data/ocread/train.txt']" \
    Eval.dataset.data_dir=train_data/ocread Eval.dataset.label_file_list="['train_data/ocread/val.txt']" \
    Train.loader.batch_size_per_card=32 Train.sampler.first_bs=32 Eval.loader.batch_size_per_card=32

## 5. Export and download

In [ ]:
!python tools/export_model.py -c {CFG} -o {COMMON} \
    Global.pretrained_model=output/ocread_rec/best_accuracy Global.save_inference_dir=ocread_rec_model
!ls -la ocread_rec_model && zip -q -r ocread_rec_model.zip ocread_rec_model
files.download('ocread_rec_model.zip')